# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hacker3code/Flyrank-AI/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis + time window

**One row means:** One row in the source fact table represents one client × content page × day observation.

**Table:** I will use `fact_content_daily_performance` for daily search and engagement measurements. I will use `dim_content` when content-level metadata such as creation date is needed.

**Time window:** I will use March 2026 (`month=2026-03`) as my development month.

**Decision frame:** I will aggregate the daily observations into content-page level features available at the decision moment and use them to support a ranked content review queue.

I will not use the `_sample` table for feature or label development because it represents the final month, June 2026, which should be treated as a sealed test month.

In [31]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Unit of analysis: client × content × day")
print("Development window: March 2026")
print("Decision: rank content pages for human review")

Unit of analysis: client × content × day
Development window: March 2026
Decision: rank content pages for human review


### Unit of analysis and time window

**One row:** One row represents one client × content page × day observation in the daily performance table.

**Time window:** I will use March 2026 (`2026-03`) as my development month.

**Decision frame:** I will aggregate the daily observations into content-page level features available at the decision moment and use them to support a ranked content review queue.

I will not use the `_sample` table for label or feature development because it represents the final month, June 2026.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields: feature / label / context / excluded

**Features:** impressions, clicks, sessions, average position, and content age. These are candidate features because they can be known before the decision is made.

**Label / proxy:** A decline indicator based on the observed trend outcome. I will treat this as a proxy for evaluating the ranking, not as proof that a page should be refreshed.

**Context:** client identifier, content identifier, reporting date, and data-availability indicators. These help identify and group observations but are not predictive features.

**Excluded:** `trend_direction` and `trend_pct` will be excluded from the honest feature set because they are used to define the decline proxy and would leak label information. Future-window performance measurements are also excluded because they would not be available at the decision moment.

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Candidate features: impressions, clicks, sessions, average position, content age")
print("Proxy: observed decline")
print("Context: client, content, date, availability")
print("Excluded: label-derived fields and future information")

Candidate features: impressions, clicks, sessions, average position, content age
Proxy: observed decline
Context: client, content, date, availability
Excluded: label-derived fields and future information


### Fields

**Features:** impressions, clicks, CTR, average position, and content age.

**Label / proxy:** A future-window content decline or recovery outcome used for evaluation.

**Context:** client identifier, content identifier, reporting date, and data-availability indicators.

**Excluded:** future performance measurements and existing product decision outputs. These are excluded because they would either leak future information or reproduce the decision we are trying to support.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification approach

I use February 2026 as the feature/development window. I verify the source grain, the number and date range of observations, and data availability before constructing the feature frame.

March 2026 is kept as the subsequent outcome window so that future information is not used to construct February features.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Define the warehouse partitions used in this notebook.

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("February feature window: ready")
print("March outcome window: ready")
# Define the February 2026 feature window

FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"

print("FEB is defined.")

February feature window: ready
March outcome window: ready
FEB is defined.


In [34]:
# QUERY 1 — Verify the source grain

grain = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT
        CAST(report_date AS VARCHAR)
        || '|' || client_hash_id
        || '|' || content_hash_id
    ) AS distinct_keys
FROM {FEB}
""").df()

display(grain)

if grain.loc[0, "row_count"] == grain.loc[0, "distinct_keys"]:
    print("Verified: one row = one report_date × client × content observation.")
else:
    print("Warning: duplicate grain keys were found.")

,row_count,distinct_keys
0,7355108,7355108


Verified: one row = one report_date × client × content observation.


The measured row count and distinct `report_date × client_hash_id × content_hash_id` key count are used to verify the daily fact-table grain. This confirms whether each source observation represents one client-content-day record.

In [35]:
# QUERY 2 — February row count and date span

date_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM {FEB}
""").df()

display(date_check)

,row_count,first_date,last_date
0,7355108,2026-02-01,2026-02-28


In [36]:
# QUERY 3 — Data availability check

availability = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,
    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows
FROM {FEB}
""").df()

display(availability)

,total_rows,gsc_available_rows,ga4_available_rows
0,7355108,2621783,145321


In [37]:
# Inspect the actual columns in the February data

schema = con.sql(f"""
SELECT *
FROM {FEB}
LIMIT 1
""").df()

print("Columns available in the February table:")
print(schema.columns.tolist())

display(schema)

Columns available in the February table:
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-02-01,client_e547b89c05043229,content_7995404695ee1ffd,True,True,True,False,57,0,1778,...,0,0,0,0,0,0,0,0,0,2026-02


### Five features and availability

1. **gsc_impressions_feb** — available at the decision moment because impressions have already been measured during February.

2. **gsc_clicks_feb** — available because Search Console clicks have already been observed during the feature window.

3. **ga4_sessions_feb** — available when GA4 data is available during February; unavailable GA4 data is not treated as zero.

4. **avg_position_feb** — available because search position was measured during the February feature window.

5. **content_age_days** — available because the content creation date is known before the decision.

All five features are restricted to the February window. March is reserved for measuring the subsequent outcome.

In [38]:
# Build five February features.
# One row = one client × content page.

feature_frame = con.sql(f"""
WITH february AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions_feb,
        SUM(gsc_clicks) AS gsc_clicks_feb,
        SUM(ga4_sessions) AS ga4_sessions_feb,
        AVG(gsc_avg_position) AS avg_position_feb

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    client_hash_id,
    content_hash_id,
    gsc_impressions_feb,
    gsc_clicks_feb,
    ga4_sessions_feb,
    avg_position_feb

FROM february
LIMIT 1000
""").df()

display(feature_frame.head(10))
print("Feature frame shape:", feature_frame.shape)

,client_hash_id,content_hash_id,gsc_impressions_feb,gsc_clicks_feb,ga4_sessions_feb,avg_position_feb
0,client_3ffa76342f366962,content_fb84747a57b8b665,0.0,0.0,NaN,NaN
1,client_3ffa76342f366962,content_feccf822ac21326e,0.0,0.0,NaN,NaN
2,client_3ffa76342f366962,content_17cf93c10413ebe9,0.0,0.0,NaN,NaN
3,client_3ffa76342f366962,content_a9905735266f8697,0.0,0.0,NaN,NaN
4,client_3ffa76342f366962,content_31c34765e7bba2f0,0.0,0.0,NaN,NaN
5,client_3ffa76342f366962,content_dcdf7e842dd640a5,0.0,0.0,NaN,NaN
6,client_3ffa76342f366962,content_dcf77b02e9c13344,0.0,0.0,NaN,NaN
7,client_3ffa76342f366962,content_3c3439c4de063402,0.0,0.0,NaN,NaN
8,client_3ffa76342f366962,content_14aa55a733a83e24,0.0,0.0,NaN,NaN
9,client_3ffa76342f366962,content_1ffdc7d0f4c3c5b8,0.0,0.0,NaN,NaN


Feature frame shape: (1000, 6)


In [39]:
# Correct content-dimension path

DIM_CONTENT = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

print(DIM_CONTENT)

hf://datasets/FlyRank/internship-warehouse/dim_content.parquet


In [40]:
# Inspect the content dimension

content_sample = con.sql(f"""
SELECT *
FROM read_parquet('{DIM_CONTENT}')
LIMIT 1
""").df()

print("dim_content columns:")
print(content_sample.columns.tolist())

display(content_sample)

dim_content columns:
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False


In [41]:
# Build the final five-feature frame.
# One row = one client × content page.
# Features are constructed only from information available by the end of February.

feature_frame = con.sql(f"""
WITH february AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS gsc_impressions_feb,
        SUM(gsc_clicks) AS gsc_clicks_feb,
        SUM(ga4_sessions) AS ga4_sessions_feb,
        AVG(gsc_avg_position) AS avg_position_feb

    FROM {FEB}

    GROUP BY
        client_hash_id,
        content_hash_id
),

content AS (
    SELECT
        client_hash_id,
        content_hash_id,
        content_created_date
    FROM read_parquet('{DIM_CONTENT}')
)

SELECT
    f.client_hash_id,
    f.content_hash_id,
    f.gsc_impressions_feb,
    f.gsc_clicks_feb,
    f.ga4_sessions_feb,
    f.avg_position_feb,

    DATE_DIFF(
        'day',
        CAST(c.content_created_date AS DATE),
        DATE '2026-02-28'
    ) AS content_age_days

FROM february f
LEFT JOIN content c
    ON f.client_hash_id = c.client_hash_id
    AND f.content_hash_id = c.content_hash_id

LIMIT 1000
""").df()

display(feature_frame.head(10))
print("Feature frame shape:", feature_frame.shape)

,client_hash_id,content_hash_id,gsc_impressions_feb,gsc_clicks_feb,ga4_sessions_feb,avg_position_feb,content_age_days
0,client_3ffa76342f366962,content_b1fc2cbd0eb808db,0.0,0.0,NaN,NaN,175
1,client_3ffa76342f366962,content_c58f11c7b33e1b01,0.0,0.0,NaN,NaN,175
2,client_3ffa76342f366962,content_d0d3d8079e4e2580,0.0,0.0,NaN,NaN,175
3,client_3ffa76342f366962,content_43147be54c74d162,0.0,0.0,NaN,NaN,175
4,client_3ffa76342f366962,content_48995646c9f4fb7a,0.0,0.0,NaN,NaN,175
5,client_3ffa76342f366962,content_1a2285833cd8de72,0.0,0.0,NaN,NaN,175
6,client_3ffa76342f366962,content_1daa35e738000391,0.0,0.0,NaN,NaN,175
7,client_3ffa76342f366962,content_2ac76486e3cdc085,0.0,0.0,NaN,NaN,175
8,client_3ffa76342f366962,content_d546a6be9a5c4ee0,0.0,0.0,NaN,NaN,175
9,client_3ffa76342f366962,content_fd42ad4db7deb553,0.0,0.0,NaN,NaN,175


Feature frame shape: (1000, 7)


In [42]:
# Create the March outcome.
# 1 = zero observed GSC clicks in March
# 0 = at least one observed GSC click in March

march_outcome = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    CASE
        WHEN SUM(gsc_clicks) = 0 THEN 1
        ELSE 0
    END AS went_dark

FROM {MAR}

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

display(march_outcome.head(10))

print("March outcome rows:", len(march_outcome))

,client_hash_id,content_hash_id,went_dark
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,1
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,0
2,client_62f4a7e64f5e0096,content_e689bc511192751a,1
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,0
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,1
5,client_62f4a7e64f5e0096,content_df22bda1218f13ff,0
6,client_62f4a7e64f5e0096,content_aa184c1b4ea518e5,1
7,client_62f4a7e64f5e0096,content_b4de71c8ef5c4791,0
8,client_62f4a7e64f5e0096,content_d7568011c4325a33,0
9,client_62f4a7e64f5e0096,content_e847a4dcc8af3742,1


March outcome rows: 176738


### Label / proxy

The March outcome is a proxy for whether a page went dark: March had zero measured GSC clicks for a page that had enough February evidence to enter the decision universe.

This is an observed outcome used to evaluate the ranking. It does not prove that the page needs a refresh or that a particular content change caused the decline.

### Deliberate leakage experiment

To demonstrate leakage, I will temporarily add a feature derived directly from the March outcome. This information would not have been available at the February decision point.

The model should become unrealistically strong when this leaked feature is included. I will then remove the feature and retain only the honest February features.

The leaked result is a demonstration of a failure mode, not a valid model result.

In [43]:
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("March data path ready.")

March data path ready.


In [44]:
# Create the March outcome.
# 1 = zero observed GSC clicks in March
# 0 = at least one observed GSC click in March

march_outcome = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,

    CASE
        WHEN SUM(gsc_clicks) = 0 THEN 1
        ELSE 0
    END AS went_dark

FROM {MAR}

WHERE gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id
""").df()

display(march_outcome.head(10))
print("March outcome rows:", len(march_outcome))

,client_hash_id,content_hash_id,went_dark
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,1
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,0
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,0
6,client_73cda7b4e4f265ea,content_1f380a642aed423b,0
7,client_73cda7b4e4f265ea,content_22c063002b7c1caf,0
8,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,0
9,client_73cda7b4e4f265ea,content_20403327d8d9374c,0


March outcome rows: 176738


In [45]:
# Combine February features with the March outcome.

frame = feature_frame.merge(
    march_outcome,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Combined frame shape:", frame.shape)
display(frame.head(10))

Combined frame shape: (142, 8)


,client_hash_id,content_hash_id,gsc_impressions_feb,gsc_clicks_feb,ga4_sessions_feb,avg_position_feb,content_age_days,went_dark
0,client_3ffa76342f366962,content_919d1fb681a61380,0.0,0.0,NaN,NaN,175,1
1,client_3ffa76342f366962,content_da44264c1fd25b4b,11.0,0.0,0.0,6.500000,174,1
2,client_3ffa76342f366962,content_6548d5911d08c81a,0.0,0.0,NaN,NaN,174,1
3,client_3ffa76342f366962,content_886058870bed99cf,0.0,0.0,NaN,NaN,174,1
4,client_3ffa76342f366962,content_f9a0be94a009ab70,31.0,1.0,0.0,13.987500,174,1
5,client_3ffa76342f366962,content_7ac4defc3d459d26,1.0,1.0,NaN,0.000000,174,1
6,client_3ffa76342f366962,content_769436a447799cc7,38.0,1.0,0.0,9.338235,174,0
7,client_3ffa76342f366962,content_05f5b637036e530c,4.0,0.0,0.0,5.166667,173,1
8,client_3ffa76342f366962,content_a04e441bd0dc8b72,0.0,0.0,NaN,NaN,173,1
9,client_3ffa76342f366962,content_cb670145328874c6,0.0,0.0,NaN,NaN,173,1


In [46]:
# DELIBERATE LEAKAGE — intentionally invalid

feature_columns = [
    "gsc_impressions_feb",
    "gsc_clicks_feb",
    "ga4_sessions_feb",
    "avg_position_feb",
    "content_age_days"
]

frame["LEAKED_MARCH_LABEL"] = frame["went_dark"]

print("Leaked feature added intentionally.")
print(
    "Does leaked feature exactly equal the label?",
    (frame["LEAKED_MARCH_LABEL"] == frame["went_dark"]).all()
)

Leaked feature added intentionally.
Does leaked feature exactly equal the label? True


In [47]:
frame = frame.drop(columns=["LEAKED_MARCH_LABEL"])

print("Leaked feature removed.")
print("Final honest feature columns:")
print(feature_columns)

display(frame[feature_columns + ["went_dark"]].head(10))

Leaked feature removed.
Final honest feature columns:
['gsc_impressions_feb', 'gsc_clicks_feb', 'ga4_sessions_feb', 'avg_position_feb', 'content_age_days']


,gsc_impressions_feb,gsc_clicks_feb,ga4_sessions_feb,avg_position_feb,content_age_days,went_dark
0,0.0,0.0,NaN,NaN,175,1
1,11.0,0.0,0.0,6.500000,174,1
2,0.0,0.0,NaN,NaN,174,1
3,0.0,0.0,NaN,NaN,174,1
4,31.0,1.0,0.0,13.987500,174,1
5,1.0,1.0,NaN,0.000000,174,1
6,38.0,1.0,0.0,9.338235,174,0
7,4.0,0.0,0.0,5.166667,173,1
8,0.0,0.0,NaN,NaN,173,1
9,0.0,0.0,NaN,NaN,173,1


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### Data limits

This data has an unbalanced history, so not every client or content page has the same amount of historical data. Some early observations are GSC-only because Analytics data was not yet available. Therefore, missing Analytics values should not automatically be treated as zero.

There can also be overlaps between the feature window and the future outcome window. I need to keep these windows separate so that future information does not leak into the features.

Because of these limitations, my results should be treated as directional decision-support rather than proof of causation or a complete picture of content performance.

## Data limits

The warehouse has an unbalanced history: different clients and content pages have different amounts of historical data.

Some observations are GSC-only because Analytics was not available for every period. Therefore, unavailable Analytics measurements should not automatically be interpreted as zero activity.

The feature and outcome windows must also remain separate. February is used for features and March for the outcome. If March information were included in the February features, the model would have access to future information and its evaluation would be misleading.

These limitations mean the results should be described as observed, measured and directional decision-support rather than causal evidence or a complete explanation of content performance.

## Data limits

This warehouse has an unbalanced history, so different clients and content pages can have different amounts of historical data.

Some observations are GSC-only because Analytics data is not available for every period. Therefore, missing GA4 measurements should not automatically be interpreted as zero activity.

The feature and outcome windows must remain separate. February is used to construct the features and March is used to observe the outcome. If March information were included in February features, the evaluation could be affected by future-information leakage.

The `went_dark` outcome is an observed proxy based on zero measured GSC clicks. It does not prove that a page is poor quality or that refreshing it will improve performance.

These limitations mean the results should be described as observed, measured and directional decision-support rather than causal evidence or a complete explanation of content performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.